# PointNet

In [1]:
from dataclasses import dataclass
from typing import List, Optional, Tuple, Union
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

from torch_pointcloud.layers.activations import get_act
from torch_pointcloud.layers.mlp import MLP

In [2]:
@dataclass
class PointNetConfig:
    """Configuration class for PointNet models"""
    num_classes: int
    in_channels: int = 3
    features_dim: int = 1024
    dropout: float = 0.5
    act: str = "relu"
    backbone_channels: List[int] = (64, 64, 64, 128, 1024)
    head_channels: List[int] = (512,)
    use_transform: bool = False
    transform_dims: List[int] = (64, 128, 1024)
    bias: bool = False


In [3]:
class TNet(nn.Module):
    """Input/Feature Transform Network"""
    def __init__(self, k: int, transform_dims: List[int], act: str = "relu"):
        super().__init__()
        self.k = k
        
        self.transform = nn.Sequential(
            *[Block(dim_in, dim_out, act=act) 
              for dim_in, dim_out in zip([k] + transform_dims[:-1], transform_dims)]
        )
        
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.regressor = MLP(
            [transform_dims[-1], 512, 256, k*k],
            act=act,
            dropout=0.0
        )

    def forward(self, x: Tensor) -> Tensor:
        batch_size = x.size(0)
        
        x = self.transform(x)
        x = self.pool(x).view(batch_size, -1)
        x = self.regressor(x).view(batch_size, self.k, self.k)
        
        # Add identity matrix for regularization
        identity = torch.eye(self.k, device=x.device).unsqueeze(0)
        x = x + identity
        return x

class Block(nn.Module):
    """Basic convolution block with optional batch norm and activation"""
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 1,
        stride: int = 1,
        bias: bool = False,
        act: str = "relu",
        use_bn: bool = True,
    ) -> None:
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, 
                            stride=stride, bias=bias)
        self.bn = nn.BatchNorm1d(out_channels) if use_bn else nn.Identity()
        self.act = get_act(act)

    def forward(self, x: Tensor) -> Tensor:
        return self.act(self.bn(self.conv(x)))

class PointNetBackbone(nn.Module):
    """Flexible PointNet backbone"""
    def __init__(
        self,
        in_channels: int,
        channels: List[int],
        act: str = "relu",
        use_transform: bool = False,
        transform_dims: List[int] = (64, 128, 1024),
        bias: bool = False,
    ) -> None:
        super().__init__()
        
        self.input_transform = TNet(in_channels, transform_dims) if use_transform else None
        
        layers = []
        current_channels = in_channels
        for out_channels in channels:
            layers.append(Block(current_channels, out_channels, act=act, bias=bias))
            current_channels = out_channels
            
        self.feature_transform = TNet(64, transform_dims) if use_transform else None
        self.layers = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tuple[Tensor, Optional[Tensor], Optional[Tensor]]:
        transform_matrix1 = None
        transform_matrix2 = None
        
        if self.input_transform is not None:
            transform_matrix1 = self.input_transform(x)
            x = torch.bmm(transform_matrix1, x)
            
        if self.feature_transform is not None:
            transform_matrix2 = self.feature_transform(x)
            x = torch.bmm(transform_matrix2, x)
            
        x = self.layers(x)
        return x, transform_matrix1, transform_matrix2

class PointNetHead(nn.Module):
    """Classification head for PointNet"""
    def __init__(
        self,
        in_features: int,
        num_classes: int,
        hidden_dims: List[int] = (512,),
        dropout: float = 0.5,
        act: str = "relu",
    ) -> None:
        super().__init__()
        self.pool = nn.AdaptiveMaxPool1d(1)
        
        mlp_dims = [in_features] + list(hidden_dims) + [num_classes]
        self.mlp = MLP(mlp_dims, act=act, dropout=dropout)

    def forward(self, x: Tensor) -> Tensor:
        x = self.pool(x).squeeze(-1)
        return self.mlp(x)

class PointNet(nn.Module):
    """Modular PointNet implementation supporting multiple variants"""
    def __init__(self, config: PointNetConfig) -> None:
        super().__init__()
        
        self.backbone = PointNetBackbone(
            in_channels=config.in_channels,
            channels=config.backbone_channels,
            act=config.act,
            use_transform=config.use_transform,
            transform_dims=config.transform_dims,
            bias=config.bias,
        )
        
        self.head = PointNetHead(
            in_features=config.backbone_channels[-1],
            num_classes=config.num_classes,
            hidden_dims=config.head_channels,
            dropout=config.dropout,
            act=config.act,
        )

    def forward_features(self, x: Tensor) -> Tuple[Tensor, Optional[Tensor], Optional[Tensor]]:
        return self.backbone(x)

    def forward(self, x: Tensor) -> Union[Tensor, Tuple[Tensor, Optional[Tensor], Optional[Tensor]]]:
        x, transform1, transform2 = self.forward_features(x)
        x = self.head(x)
        
        if self.training and (transform1 is not None or transform2 is not None):
            return x, transform1, transform2
        return x

In [4]:
def pointnet_base(num_classes: int, **kwargs) -> PointNet:
    """PointNet base variant"""
    config = PointNetConfig(
        num_classes=num_classes,
        backbone_channels=[64, 64, 64, 128, 1024],
        head_channels=[512],
        **kwargs
    )
    return PointNet(config)

def pointnet_light(num_classes: int, **kwargs) -> PointNet:
    """Lightweight PointNet variant"""
    config = PointNetConfig(
        num_classes=num_classes,
        backbone_channels=[32, 32, 64, 128, 512],
        head_channels=[256],
        **kwargs
    )
    return PointNet(config)

def pointnet_deep(num_classes: int, **kwargs) -> PointNet:
    """Deeper PointNet variant"""
    config = PointNetConfig(
        num_classes=num_classes,
        backbone_channels=[64, 64, 128, 128, 256, 256, 512, 1024],
        head_channels=[512, 256],
        **kwargs
    )
    return PointNet(config)

In [5]:
model = pointnet_base(10)

In [6]:
x = torch.randn(32, 3, 1024)
out = model(x)

In [7]:
out.shape

torch.Size([32, 10])

---

In [8]:
from dataclasses import dataclass
from typing import List, Optional, Tuple, Union
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch_scatter import scatter_max, scatter_mean

In [9]:
@dataclass
class PointNetConfig:
    """Configuration class for PointNet models"""
    num_classes: int
    in_channels: int = 3
    features_dim: int = 1024
    dropout: float = 0.5
    act: str = "relu"
    backbone_channels: List[int] = (64, 64, 64, 128, 1024)
    head_channels: List[int] = (512,)
    use_transform: bool = False
    transform_dims: List[int] = (64, 128, 1024)
    bias: bool = False

In [10]:
class PackedPointCloud:
    """
    Helper class for packed point cloud format.
    Stores points as (N, D) tensor where N is total number of points across batch,
    along with batch indices indicating which batch each point belongs to.
    """
    def __init__(
        self,
        pos: Tensor,  # (N, 3) positions
        x: Optional[Tensor] = None,  # (N, C) features
        batch: Optional[Tensor] = None,  # (N,) batch indices
    ):
        self.pos = pos
        self.x = x if x is not None else pos.new_ones((pos.size(0), 1))
        
        if batch is None:
            # Assume single batch if not provided
            batch = pos.new_zeros(pos.size(0), dtype=torch.long)
        self.batch = batch
        
        # Compute batch size and points per batch
        self.batch_size = int(batch.max()) + 1 if len(batch) > 0 else 0
        self.counts = torch.bincount(batch)
        
    @staticmethod
    def from_dense(
        pos: Tensor,  # (B, N, 3)
        x: Optional[Tensor] = None,  # (B, N, C)
    ) -> 'PackedPointCloud':
        """Convert dense/padded format to packed format"""
        batch_size, num_points, _ = pos.shape
        device = pos.device
        
        # Create batch indices
        batch = torch.arange(batch_size, device=device).view(-1, 1).repeat(1, num_points)
        batch = batch.view(-1)
        
        # Reshape points and features
        pos = pos.view(-1, pos.size(-1))
        if x is not None:
            x = x.view(-1, x.size(-1))
            
        return PackedPointCloud(pos, x, batch)
    
    def to_dense(
        self,
        pad_value: float = 0.0
    ) -> Tuple[Tensor, Optional[Tensor]]:
        """Convert packed format to dense/padded format"""
        if self.batch_size == 0:
            return self.pos.unsqueeze(0), self.x.unsqueeze(0)
            
        # Get max points per batch
        max_points = int(self.counts.max())
        
        # Initialize dense tensors
        dense_pos = self.pos.new_full((self.batch_size, max_points, self.pos.size(1)), pad_value)
        dense_x = self.x.new_full((self.batch_size, max_points, self.x.size(1)), pad_value)
        
        # Fill dense tensors
        cum_counts = torch.cat([self.counts.new_zeros(1), self.counts.cumsum(0)[:-1]])
        for i in range(self.batch_size):
            start = cum_counts[i]
            length = self.counts[i]
            dense_pos[i, :length] = self.pos[start:start + length]
            dense_x[i, :length] = self.x[start:start + length]
            
        return dense_pos, dense_x

In [11]:
class PackedBlock(nn.Module):
    """Convolution block for packed point clouds"""
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 1,
        stride: int = 1,
        bias: bool = False,
        act: str = "relu",
    ) -> None:
        super().__init__()
        self.conv = nn.Linear(in_channels, out_channels, bias=bias)  # Using Linear instead of Conv1d
        self.bn = nn.BatchNorm1d(out_channels)
        self.act = get_act(act)

    def forward(self, x: Tensor) -> Tensor:
        # Input: (N, C), Output: (N, C')
        x = self.conv(x)     # (N, C')
        x = self.bn(x)
        return self.act(x)

class PackedPointNet(nn.Module):
    """PointNet for packed point clouds"""
    def __init__(self, config: PointNetConfig) -> None:
        super().__init__()
        self.config = config

        # Input transform
        self.input_transform = None
        if config.use_transform:
            self.input_transform = TNet(
                k=3,
                transform_dims=config.transform_dims,
                act=config.act
            )

        # Backbone
        backbone_layers = []
        # If using positions only, in_channels is 3
        # If using positions + features, in_channels is sum of both
        in_channels = config.in_channels  
        
        for out_channels in config.backbone_channels:
            backbone_layers.append(
                PackedBlock(
                    in_channels,
                    out_channels,
                    bias=config.bias,
                    act=config.act
                )
            )
            in_channels = out_channels
            
        self.backbone = nn.Sequential(*backbone_layers)

        # Classification head
        head_dims = [config.backbone_channels[-1]] + list(config.head_channels) + [config.num_classes]
        head_layers = []
        
        for i in range(len(head_dims) - 1):
            head_layers.extend([
                nn.Linear(head_dims[i], head_dims[i + 1]),
                nn.BatchNorm1d(head_dims[i + 1]),
                get_act(config.act)
            ])
            if i < len(head_dims) - 2 and config.dropout > 0:
                head_layers.append(nn.Dropout(config.dropout))
                
        self.head = nn.Sequential(*head_layers)

    def forward_features(
        self,
        data: PackedPointCloud
    ) -> Tuple[Tensor, Tensor]:
        x = data.x
        pos = data.pos
        batch = data.batch
        
        # Apply input transform if needed
        if self.input_transform is not None:
            trans = self.input_transform(pos, batch)
            pos = torch.bmm(
                pos.view(-1, 1, 3),
                trans
            ).view(-1, 3)
            
        # Combine features and positions
        if self.config.in_channels == 3:
            # Use positions only
            x = pos
        else:
            # Concatenate positions and features
            x = torch.cat([pos, x], dim=-1)
            
        # Apply backbone
        x = self.backbone(x)  # (N, C)
        
        # Global pooling
        x_pooled = scatter_max(x, batch, dim=0)[0]  # (B, C)
        
        return x, x_pooled

    def forward(
        self,
        pos: Tensor,
        x: Optional[Tensor] = None,
        batch: Optional[Tensor] = None
    ) -> Tensor:
        # Create packed point cloud
        data = PackedPointCloud(pos, x, batch)
        
        # Forward pass
        _, x_pooled = self.forward_features(data)
        
        # Classification head
        return self.head(x_pooled)
    

In [12]:

# Factory functions for different variants
def pointnet_tiny_packed(num_classes: int, **kwargs) -> PackedPointNet:
    """Tiny PointNet variant with packed format"""
    config = PointNetConfig(
        num_classes=num_classes,
        backbone_channels=[32, 32, 64, 128, 512],
        head_channels=[256],
        features_dim=512,
        dropout=0.3,
        **kwargs
    )
    return PackedPointNet(config)

def pointnet_base_packed(num_classes: int, **kwargs) -> PackedPointNet:
    """Standard PointNet with packed format"""
    config = PointNetConfig(
        num_classes=num_classes,
        **kwargs
    )
    return PackedPointNet(config)

def pointnet_wide_packed(num_classes: int, **kwargs) -> PackedPointNet:
    """Wide PointNet variant with packed format"""
    config = PointNetConfig(
        num_classes=num_classes,
        backbone_channels=[128, 128, 128, 256, 1024],
        head_channels=[512],
        features_dim=1024,
        **kwargs
    )
    return PackedPointNet(config)

In [13]:
class PointNetDataset:
    """Example dataset class for packed point clouds"""
    def __init__(
        self,
        points: List[Tensor],
        features: Optional[List[Tensor]] = None,
        labels: Optional[List[int]] = None
    ):
        self.points = points
        self.features = features
        self.labels = labels

    def __len__(self) -> int:
        return len(self.points)

    def __getitem__(self, idx: int) -> Tuple[Tensor, Optional[Tensor], Optional[int]]:
        point = self.points[idx]
        feature = None if self.features is None else self.features[idx]
        label = None if self.labels is None else self.labels[idx]
        return point, feature, label
    
    
def packed_collate(
    batch: List[Tuple[Tensor, Optional[Tensor], Optional[int]]]
) -> Tuple[PackedPointCloud, Optional[Tensor]]:
    """Collate function for packed point clouds"""
    points, features, labels = list(zip(*batch))
    
    # Concatenate points and create batch indices
    batch_idx = []
    for i, p in enumerate(points):
        batch_idx.append(torch.full((len(p),), i, dtype=torch.long))
    batch_idx = torch.cat(batch_idx)
    points = torch.cat(points)
    
    # Concatenate features if present
    if features[0] is not None:
        features = torch.cat(features)
    else:
        features = None
        
    # Create packed point cloud
    packed = PackedPointCloud(points, features, batch_idx)
    
    # Convert labels to tensor if present
    if labels[0] is not None:
        labels = torch.tensor(labels)
    else:
        labels = None
        
    return packed, labels

In [15]:
import torch

# Create random point cloud with varying number of points
num_samples = 3
points_per_sample = [100, 150, 120]  # Variable number of points

# Create points and features
points = [torch.randn(n, 3) for n in points_per_sample]  # Each point has 3D coordinates
features = [torch.randn(n, 6) for n in points_per_sample]  # Each point has 6 additional features
labels = list(range(num_samples))

dataset = PointNetDataset(points, features, labels)

# Create dataloader
from torch.utils.data import DataLoader

dataloader = DataLoader(
    dataset,
    batch_size=3,
    collate_fn=packed_collate,
    shuffle=True
)

# Create model - specify in_channels as sum of position dims (3) and feature dims (6)
model = pointnet_base_packed(num_classes=10, in_channels=9)  # 3 + 6 = 9 channels

# Forward pass
for data, labels in dataloader:
    output = model(data.pos, data.x, data.batch)
    print(f"Input points shape: {data.pos.shape}")  # [total_points, 3]
    print(f"Input features shape: {data.x.shape}")  # [total_points, 6]
    print(f"Output shape: {output.shape}")  # [batch_size, num_classes]
    break

Input points shape: torch.Size([370, 3])
Input features shape: torch.Size([370, 6])
Output shape: torch.Size([3, 10])


---

In [16]:
from dataclasses import dataclass
from typing import List, Optional, Tuple, Union
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

@dataclass
class TNetConfig:
    """Configuration for Transform Network"""
    k: int
    channels: List[int]  # [64, 128, 1024]
    mlp_channels: List[int]  # [512, 256]
    bias: bool = False
    act: str = "relu"

@dataclass
class PointNetConfig:
    """Configuration for PointNet"""
    num_classes: int
    in_channels: int = 3
    use_input_transform: bool = True
    use_feature_transform: bool = True
    input_transform: Optional[TNetConfig] = None
    feature_transform: Optional[TNetConfig] = None
    backbone_channels: List[int] = (64, 64, 64, 128, 1024)
    head_channels: List[int] = (512, 256)
    dropout: float = 0.5
    act: str = "relu"
    bias: bool = False

class Block(nn.Module):
    """Basic building block with optional batch norm and activation"""
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 1,
        stride: int = 1,
        bias: bool = False,
        act: str = "relu",
    ) -> None:
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, bias=bias)
        self.bn = nn.BatchNorm1d(out_channels)
        self.act = get_act(act)

    def forward(self, x: Tensor) -> Tensor:
        return self.act(self.bn(self.conv(x)))

class TNet(nn.Module):
    """Transform Network using Blocks"""
    def __init__(self, config: TNetConfig):
        super().__init__()
        self.k = config.k
        
        # Feature extraction using Blocks
        layers = []
        in_channels = config.k
        for out_channels in config.channels:
            layers.append(
                Block(
                    in_channels, 
                    out_channels,
                    bias=config.bias,
                    act=config.act
                )
            )
            in_channels = out_channels
            
        self.features = nn.Sequential(*layers)
        
        # MLP for transformation matrix
        mlp_channels = [config.channels[-1]] + config.mlp_channels + [config.k * config.k]
        mlp_layers = []
        
        for i in range(len(mlp_channels) - 1):
            is_last = i == len(mlp_channels) - 2
            mlp_layers.extend([
                nn.Linear(mlp_channels[i], mlp_channels[i + 1], bias=True),
                nn.Identity() if is_last else nn.BatchNorm1d(mlp_channels[i + 1]),
                nn.Identity() if is_last else get_act(config.act)
            ])
            
        self.mlp = nn.Sequential(*mlp_layers)
        
        # Initialize last layer to identity transform
        with torch.no_grad():
            last_layer = self.mlp[-3]  # Get last linear layer
            last_layer.weight.zero_()
            iden = torch.eye(config.k, dtype=torch.float32)
            last_layer.bias.data.copy_(iden.view(-1))

    def forward(self, x: Tensor) -> Tensor:
        batch_size = x.size(0)
        
        # Ensure input is in (B, C, N) format
        if x.size(1) != self.k:
            x = x.transpose(1, 2)
            
        # Feature extraction
        x = self.features(x)
        
        # Global max pooling
        x = torch.max(x, dim=2, keepdim=False)[0]
        
        # MLP to transformation matrix
        x = self.mlp(x)
        x = x.view(batch_size, self.k, self.k)
        
        # Add identity
        identity = torch.eye(self.k, dtype=x.dtype, device=x.device)
        x = x + identity.unsqueeze(0)
        
        return x

class PointNet(nn.Module):
    """Flexible PointNet implementation"""
    def __init__(self, config: PointNetConfig):
        super().__init__()
        self.config = config

        # Input transform
        self.input_transform = None
        if config.use_input_transform:
            tnet_config = config.input_transform or TNetConfig(
                k=3,
                channels=[64, 128, 1024],
                mlp_channels=[512, 256],
                bias=config.bias,
                act=config.act
            )
            self.input_transform = TNet(tnet_config)

        # Build backbone
        backbone_layers = []
        in_channels = config.in_channels
        
        for i, out_channels in enumerate(config.backbone_channels):
            # Add feature transform after first few layers if specified
            if i == 2 and config.use_feature_transform:  # After second 64-channel layer
                tnet_config = config.feature_transform or TNetConfig(
                    k=64,
                    channels=[64, 128, 1024],
                    mlp_channels=[512, 256],
                    bias=config.bias,
                    act=config.act
                )
                self.feature_transform = TNet(tnet_config)
            
            backbone_layers.append(
                Block(
                    in_channels,
                    out_channels,
                    bias=config.bias,
                    act=config.act
                )
            )
            in_channels = out_channels
            
        self.backbone = nn.Sequential(*backbone_layers)

        # Global pooling
        self.global_pool = nn.AdaptiveMaxPool1d(1)

        # Classification head
        head_dims = [config.backbone_channels[-1]] + list(config.head_channels) + [config.num_classes]
        head_layers = []
        
        for i in range(len(head_dims) - 1):
            is_last = i == len(head_dims) - 2
            head_layers.extend([
                nn.Linear(head_dims[i], head_dims[i + 1], bias=True),
                nn.Identity() if is_last else nn.BatchNorm1d(head_dims[i + 1]),
                nn.Identity() if is_last else get_act(config.act)
            ])
            if not is_last and config.dropout > 0:
                head_layers.append(nn.Dropout(config.dropout))
                
        self.head = nn.Sequential(*head_layers)

    def forward_features(self, x: Tensor) -> Tuple[Tensor, Optional[Tensor], Optional[Tensor]]:
        input_trans = feat_trans = None
        
        # Input transform
        if self.input_transform is not None:
            input_trans = self.input_transform(x)
            x = torch.bmm(x.transpose(2, 1), input_trans).transpose(2, 1)
        
        # Initial layers
        for i, layer in enumerate(self.backbone):
            # Apply feature transform after second 64-channel layer
            if i == 2 and hasattr(self, 'feature_transform'):
                feat_trans = self.feature_transform(x)
                x = torch.bmm(x.transpose(2, 1), feat_trans).transpose(2, 1)
            x = layer(x)
        
        return x, input_trans, feat_trans

    def forward(self, x: Tensor) -> Union[Tensor, Tuple[Tensor, Tensor, Tensor]]:
        x, input_trans, feat_trans = self.forward_features(x)
        x = self.global_pool(x).squeeze(-1)
        x = self.head(x)
        
        if self.training and (input_trans is not None or feat_trans is not None):
            return x, input_trans, feat_trans
        return x

def get_original_pointnet_config(num_classes: int) -> PointNetConfig:
    """Get config matching original PointNet paper"""
    return PointNetConfig(
        num_classes=num_classes,
        in_channels=3,
        use_input_transform=True,
        use_feature_transform=True,
        backbone_channels=[64, 64, 64, 128, 1024],
        head_channels=[512, 256],
        dropout=0.3,
        act="relu",
        bias=False
    )

# Factory functions for different variants
def pointnet_original(num_classes: int) -> PointNet:
    """Original PointNet architecture"""
    return PointNet(get_original_pointnet_config(num_classes))

def pointnet_simple(num_classes: int) -> PointNet:
    """Simplified PointNet without transforms"""
    config = PointNetConfig(
        num_classes=num_classes,
        use_input_transform=False,
        use_feature_transform=False,
        backbone_channels=[64, 64, 128, 256],
        head_channels=[256],
        dropout=0.5
    )
    return PointNet(config)

def pointnet_deep(num_classes: int) -> PointNet:
    """Deeper PointNet variant"""
    config = PointNetConfig(
        num_classes=num_classes,
        backbone_channels=[64, 64, 128, 128, 256, 256, 512, 1024],
        head_channels=[512, 256],
        dropout=0.5
    )
    return PointNet(config)

In [17]:
def feature_transform_regularizer(trans: Tensor) -> Tensor:
    """
    Compute regularization loss for feature transform matrix.
    Enforces the transformation to be orthogonal by penalizing deviation from I = T*T^T
    
    Args:
        trans: Transformation matrix of shape (B, K, K)
        
    Returns:
        loss: Regularization loss (scalar)
    """
    d = trans.size()[1]
    I = torch.eye(d, device=trans.device)[None, :, :]
    # Compute deviation from orthogonality
    loss = torch.mean(torch.norm(torch.bmm(trans, trans.transpose(2, 1)) - I, dim=(1, 2)))
    return loss

class PointNet(nn.Module):
    """PointNet with feature transform regularization"""
    def __init__(self, config: PointNetConfig):
        super().__init__()
        self.config = config
        
        # ... rest of the initialization code remains the same ...

    def get_regularization_loss(self, feat_trans: Optional[Tensor]) -> Tensor:
        """
        Compute feature transform regularization loss if applicable
        
        Args:
            feat_trans: Feature transformation matrix or None
            
        Returns:
            loss: Regularization loss (0 if no feature transform)
        """
        if feat_trans is not None:
            return feature_transform_regularizer(feat_trans)
        return torch.tensor(0.0, device=feat_trans.device if feat_trans is not None else 'cpu')

    def forward(self, x: Tensor) -> Union[Tensor, Tuple[Tensor, Tensor]]:
        """
        Forward pass returning both prediction and regularization loss
        
        Args:
            x: Input point cloud (B, N, 3) or (B, 3, N)
            
        Returns:
            outputs: Model predictions
            reg_loss: Regularization loss (if in training mode)
        """
        x, input_trans, feat_trans = self.forward_features(x)
        x = self.global_pool(x).squeeze(-1)
        x = self.head(x)
        
        if self.training:
            reg_loss = self.get_regularization_loss(feat_trans)
            return x, reg_loss
        return x

# Example training loop
def train_pointnet(model: PointNet, train_loader, optimizer, criterion, reg_weight: float = 0.001):
    """
    Training loop incorporating feature transform regularization
    
    Args:
        model: PointNet model
        train_loader: DataLoader for training data
        optimizer: PyTorch optimizer
        criterion: Classification loss criterion
        reg_weight: Weight for regularization loss (default: 0.001 as in paper)
    """
    model.train()
    
    for points, target in train_loader:
        optimizer.zero_grad()
        
        # Forward pass
        pred, reg_loss = model(points)
        
        # Compute total loss
        cls_loss = criterion(pred, target)
        total_loss = cls_loss + reg_weight * reg_loss
        
        # Backward pass
        total_loss.backward()
        optimizer.step()
        
        # Optional: log losses
        print(f"Classification loss: {cls_loss.item():.4f}, "
              f"Regularization loss: {reg_loss.item():.4f}")

In [18]:
# Create model and move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = pointnet_original(num_classes=40).to(device)

x = torch.randn(32, 3, 1024).to(device)
out, reg_loss = model(x)

AttributeError: 'PointNet' object has no attribute 'forward_features'